# Analysis Main Notebook

이 노트북은 `Analysis` 안의 분석 도구를 한 곳에서 호출합니다. SSH 환경에서도 결과를 볼 수 있도록 그래프는 `trends`, JSON과 최종 표 CSV는 `final` 아래에 저장합니다.

## 0. 공통 설정

In [33]:
from pathlib import Path
import json
import pandas as pd
import sys

# Determine repository root by climbing from the current working directory.
# This makes imports and relative result/output paths work from the notebook
# even if the notebook server starts from a subdirectory.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from Analysis.analyzer import Analyzer
from Analysis.distance_metrics import nearestStats
from Analysis.statistics import calcStats, printStats, reportCluster
from Analysis.result_io import finalPoints, listBands, loadAlgoRuns
from Analysis.trends import (
    plotConverge,
    saveReport,
    coverSummary,
)

RESULTS_ROOT = REPO_ROOT / "__RESULTS__"
ALGORITHM = "greedy"          # ga, pso, drl, greedy
MAP_NAME = "gangjin.down" # gangjin.down, seocho.up, ...
SEED_BAND = "40-60"      # None이면 전체 band, 단일 run 분석에는 하나의 band 사용
GRID_M = 5.0
COVERAGE = 45
TARGET_VALUES = (2, 3)

# 구조화된 출력 디렉토리 설정: analysis/{algorithm}/{map_name}/
ANALYSIS_ROOT = RESULTS_ROOT / "analysis"
ALGO_MAP_DIR = ANALYSIS_ROOT / ALGORITHM / MAP_NAME.replace("/", "_")
TRENDS_DIR = ALGO_MAP_DIR / "trends"
FINAL_DIR = ALGO_MAP_DIR / "final"
REPORT_DIR = TRENDS_DIR

# 디렉토리 생성
TRENDS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

RUN_DIR = RESULTS_ROOT / ALGORITHM / MAP_NAME / SEED_BAND if SEED_BAND else RESULTS_ROOT / ALGORITHM / MAP_NAME
RUN_DIR


PosixPath('/workspace/__RESULTS__/greedy/gangjin.down/40-60')

## 1. 결과 파일과 seed band 확인

In [34]:
bands = listBands(results_root=RESULTS_ROOT, algorithm=ALGORITHM, map_name=MAP_NAME)
records = loadAlgoRuns(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    seed_band=SEED_BAND,
)
print("seed bands:", bands)
print("run count in selected band:", len(records))
records[0][0]

seed bands: ['40-60', '60-80', '80-100', '100-120', '120-140']
run count in selected band: 10


PosixPath('/workspace/__RESULTS__/greedy/gangjin.down/40-60/20260609_194325.json')

## 2. 단일 run 세대별 추이

`Analyzer`는 하나의 JSON run에 대해 센서 수, 커버리지, fitness 추이를 확인합니다.

In [35]:
analyzer = Analyzer(result_root_path=str(RUN_DIR), file_path=0)

safe_map = MAP_NAME.replace("/", "_").replace(".", "_")

analyzer.plot_evolution_trend(
    xtick_step=50,
    include_corners=True,
    figsize=(10, 3),
    dpi=150,
    show=False,
    save_path=TRENDS_DIR / f"{safe_map}_sensor_count.png",
    close=True,
)
analyzer.plot_coverage_trend(
    xtick_step=50,
    figsize=(10, 3),
    dpi=150,
    show=False,
    save_path=TRENDS_DIR / f"{safe_map}_coverage.png",
    close=True,
)
analyzer.plot_fitness_trend(
    xtick_step=50,
    figsize=(10, 3),
    dpi=150,
    show=False,
    save_path=TRENDS_DIR / f"{safe_map}_fitness.png",
    close=True,
)
print("saved single-run plots to", TRENDS_DIR)


saved single-run plots to /workspace/__RESULTS__/analysis/greedy/gangjin.down/trends


## 3. 선택 band의 기본 통계

In [36]:
printStats(str(RUN_DIR))
stats_tuple = calcStats(str(RUN_DIR))
stats_tuple

ValueError: run has insufficient generations: 17

## 4. 센서 간 거리 분석

In [ ]:
reportCluster(str(RUN_DIR), MAP_NAME, grid_m=GRID_M)

distance_rows = []
for path, run in records:
    d = nearestStats(finalPoints(run), grid_m=GRID_M)
    d["run_name"] = run.get("run_name", path.stem)
    distance_rows.append(d)
distance_df = pd.DataFrame(distance_rows).set_index("run_name")
distance_df.round(3).head()

[gangjin.down] total runs: 10
[final] 평균 군집거리 mean ± std: 56.611 ± 0.000 m
[final] 군집거리 min / max: 56.611 / 56.611 m


,mean_m,std_m,min_m,max_m,n_points
run_name,,,,,
20260609_195605,56.611,9.454,46.098,80.0,19
20260609_195619,56.611,9.454,46.098,80.0,19
20260609_195633,56.611,9.454,46.098,80.0,19
20260609_195647,56.611,9.454,46.098,80.0,19
20260609_195701,56.611,9.454,46.098,80.0,19


## 5. 초기 seed 센서수별 수렴 분석

`metric='best'`는 세대별 best solution 센서수를 기준으로 수렴을 봅니다. `metric='avg'`로 바꾸면 로그의 평균 센서 수 기준으로 분석합니다.

In [ ]:
safe_map = MAP_NAME.replace("/", "_").replace(".", "_")
convergence = plotConverge(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    seed_bands=bands,
    include_corners=True,
    metric="best",
    threshold=0.5,
    dpi=300,
    show=False,
    save_path=TRENDS_DIR / f"convergence_{safe_map}.png",
)
print("convergence generation:", convergence["convergence"]["convergence_gen"])
convergence["convergence"]


convergence generation: 79


{'convergence_gen': 79,
 'by_band': {'40-60': {'gen_from': 78,
   'max_change': 2.0,
   'abs_diffs': [0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    2.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    1.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    2.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    1.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0]},
  '60-80'

## 6. 전체 커버리지와 중첩 커버리지 분석

In [ ]:
report = saveReport(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    output_dir=REPORT_DIR,
    summary_dir=FINAL_DIR,
    seed_bands=bands,
    include_corners=True,
    metric="best",
    threshold=0.5,
    target_values=TARGET_VALUES,
    dpi=300,
    show=False,
)
print(json.dumps({k: report[k] for k in ["convergence_plot", "coverage_overlap_plot", "coverage_overlap_summary"]}, indent=2))


{
  "convergence_plot": "/workspace/__RESULTS__/analysis/drl/gangjin.down/trends/sensor_convergence.png",
  "coverage_overlap_plot": "/workspace/__RESULTS__/analysis/drl/gangjin.down/trends/coverage_overlap.png",
  "coverage_overlap_summary": "/workspace/__RESULTS__/analysis/drl/gangjin.down/final/coverage_overlap.json"
}


## 7. 논문 표용 요약 CSV

In [ ]:
summary = coverSummary(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    seed_bands=bands,
    target_values=TARGET_VALUES,
)
summary_df = pd.DataFrame.from_dict(summary, orient="index")
safe_map = MAP_NAME.replace("/", "_").replace(".", "_")
csv_path = FINAL_DIR / "summary.csv"
summary_df.to_csv(csv_path, encoding="utf-8-sig")
print("saved", csv_path)
summary_df[[
    "runs",
    "n_sensors_mean",
    "coverage_percent_mean",
    "overlap_percent_of_target_mean",
    "overlap_percent_of_covered_mean",
    "redundant_hit_percent_mean",
    "logged_final_coverage_mean",
]].round(3)


saved /workspace/__RESULTS__/analysis/drl/gangjin.down/final/summary.csv


,runs,n_sensors_mean,coverage_percent_mean,overlap_percent_of_target_mean,overlap_percent_of_covered_mean,redundant_hit_percent_mean,logged_final_coverage_mean
40-60,10.0,19.0,99.805,54.869,54.976,38.512,99.805
60-80,10.0,20.0,99.903,60.662,60.721,42.230,99.903
80-100,10.0,19.0,99.951,59.542,59.571,39.242,99.903
100-120,10.0,20.0,99.513,58.861,59.149,41.180,99.513
120-140,10.0,20.0,99.124,58.179,58.694,40.485,99.124
